# Notebook 01 — Ingestão dos dados ANP no Lakehouse (Fabric)

Lê os CSVs consolidados (saída do ETL local ou shortcut do OneLake/ADLS) e grava
como tabelas Delta no Lakehouse da Fabric.

**Pré-requisitos**
- Lakehouse anexado ao notebook (default: `lh_anp_petroleo`)
- Arquivos em `Files/dados_brutos/` do Lakehouse:
  - `fato_mensal_2020_2025.csv`
  - `fato_anual_2020_2025.csv`
  - `dim_campos_fase_anual.csv`
  - `dim_campos.csv`

In [ ]:
from pyspark.sql.functions import col, to_date

BASE = "Files/dados_brutos"

ARQUIVOS = {
    "fato_mensal": (f"{BASE}/fato_mensal_2020_2025.csv", True),
    "fato_anual": (f"{BASE}/fato_anual_2020_2025.csv", True),
    "dim_fases_campos": (f"{BASE}/dim_campos_fase_anual.csv", True),
    "dim_campos": (f"{BASE}/dim_campos.csv", True),
}

for nome, (caminho, tem_cabecalho) in ARQUIVOS.items():
    df = (
        spark.read.option("header", str(tem_cabecalho).lower())
        .option("encoding", "UTF-8")
        .csv(caminho)
    )
    if nome == "fato_mensal":
        df = (
            df.withColumn("DATA", to_date(col("DATA")))
            .withColumn("ANO", col("ANO").cast("int"))
            .withColumn("TRIMESTRE", col("TRIMESTRE").cast("int"))
            .withColumn("MES_NUM", col("MES_NUM").cast("int"))
        )
    for c in df.columns:
        if c.endswith("BARRIS") or c.endswith("PCT"):
            df = df.withColumn(c, col(c).cast("double"))
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(nome)
    print(f"Tabela Delta criada/atualizada: {nome} ({df.count()} linhas)")

## Validação rápida

In [ ]:
display(spark.table("fato_mensal").groupBy("ANO").sum("PRODUCAO_BARRIS").orderBy("ANO"))